# Evaluación de Resultados

En esta parte revisamos si la política aprendida por el bandit tiene sentido frente al mejor canal histórico y frente a un agente global mas simple.


## Qué queremos mirar

- que canal termina recomendando el bandit por contexto
- si coincide o no con el mejor canal histórico
- cuanto regret aproximado queda
- si un agente contextual parece mejor idea que uno solo global


In [1]:
from pathlib import Path
from pprint import pprint
import sys

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.evaluacion.experimentos import ejecutar_flujo_completo


## Comenzamos ejecutando el flujo que construimos

Pero, qué hace `ejecutar_flujo_completo`?

Esta función corre todo el pipeline principal del proyecto de punta a punta. En orden, hace esto:

- carga el csv original
- limpia y tipa las columnas
- crea el contexto `audiencia + objetivo`
- arma la tabla agregada por contexto y canal
- prepara las observaciones para el bandit
- entrena los agentes contextuales
- entrena tambien un agente global para comparar
- evalúa la política aprendida
- devuelve varios resultados ya listos en un diccionario

En otras palabras, esta función nos evita repetir todo el flujo en cada notebook y deja la evaluación bastante ordenada.


In [2]:
resultado = ejecutar_flujo_completo(ROOT / 'social_media_ads_filtered.csv')

print('Estructura general de resultados:')
pprint({clave: type(valor).__name__ for clave, valor in resultado.items()})

print('Metricas finales:')
pprint(resultado['metricas_finales'])


Entrenando contexto: Women 45-60 | Brand Awareness
Entrenando contexto: Men 25-34 | Product Launch
Entrenando contexto: Men 18-24 | Brand Awareness
Entrenando contexto: Men 35-44 | Market Expansion
Entrenando contexto: Women 25-34 | Increase Sales
Entrenando contexto: Women 35-44 | Brand Awareness
Entrenando contexto: Men 25-34 | Increase Sales
Entrenando contexto: Women 45-60 | Product Launch
Entrenando contexto: Women 35-44 | Market Expansion
Entrenando contexto: Men 45-60 | Market Expansion
Entrenando contexto: Women 18-24 | Product Launch
Entrenando contexto: Women 18-24 | Brand Awareness
Entrenando contexto: Men 35-44 | Increase Sales
Entrenando contexto: Men 45-60 | Increase Sales
Entrenando contexto: Women 35-44 | Product Launch
Entrenando contexto: Women 25-34 | Brand Awareness
Entrenando contexto: Men 45-60 | Brand Awareness
Entrenando contexto: Men 35-44 | Brand Awareness
Entrenando contexto: Men 18-24 | Increase Sales
Entrenando contexto: Women 25-34 | Product Launch
Entrena

## Métricas finales

Acá se resume la evaluación general. En esta ejecución se entrenaron `36` contextos y la accuracy del mejor canal quedó en `0.75`, o sea que en 3 de cada 4 contextos el bandit terminó recomendando el mismo canal que el mejor histórico observado.

Tambien se ve que el `roi_promedio_bandit` quedó muy cerca del `roi_promedio_optimo`, y que el `regret_aproximado_promedio` es relativamente bajo. Eso sugiere que, aunque no acierta siempre exactamente en el canal ganador, sí queda bastante cerca en rendimiento.


In [3]:
resultado['evaluacion'].head(20)


,Contexto,canal_recomendado_bandit,mejor_canal_historico,acierto_mejor_canal,roi_promedio_bandit,roi_promedio_optimo,regret_aproximado
0,Women 45-60 | Brand Awareness,Facebook,Facebook,1,4.637473,4.584400,0.000000
1,Men 25-34 | Product Launch,Instagram,Instagram,1,4.060370,3.954828,0.000000
2,Men 18-24 | Brand Awareness,Facebook,Facebook,1,4.124737,4.187500,0.062763
3,Men 35-44 | Market Expansion,Instagram,Instagram,1,4.069677,3.981667,0.000000
4,Women 25-34 | Increase Sales,Instagram,Instagram,1,4.500357,4.499091,0.000000
5,Women 35-44 | Brand Awareness,Twitter,Twitter,1,4.698088,4.631538,0.000000
6,Men 25-34 | Increase Sales,Instagram,Instagram,1,4.361176,4.347586,0.000000
7,Women 45-60 | Product Launch,Twitter,Twitter,1,4.513297,4.500938,0.000000
8,Women 35-44 | Market Expansion,Instagram,Instagram,1,4.549500,4.455429,0.000000
9,Men 45-60 | Market Expansion,Facebook,Facebook,1,4.268644,4.247407,0.000000


**Qué podemos concluir de la tabla anterior?**

Esta tabla compara, por contexto, el canal recomendado por el bandit contra el mejor canal histórico.

Las columnas mas importantes son:

- `canal_recomendado_bandit`
- `mejor_canal_historico`
- `acierto_mejor_canal`
- `roi_promedio_bandit`
- `roi_promedio_optimo`
- `regret_aproximado`

Lo interesante aca es que incluso cuando no coincide exactamente el canal, a veces la diferencia de ROI no es tan grande.


In [4]:
resultado['tabla_politica_brazos'].head(20)


,canal,impresiones,recompensa_promedio,roi_promedio_observado,alpha,beta,muestra_esperada,Contexto
0,Facebook,91,0.579684,4.637473,57.0,36.0,0.612903,Women 45-60 | Brand Awareness
1,Twitter,15,0.452417,3.619333,7.0,10.0,0.411765,Women 45-60 | Brand Awareness
2,Instagram,6,0.396875,3.175000,2.0,6.0,0.250000,Women 45-60 | Brand Awareness
3,Pinterest,5,0.081687,0.653500,1.0,6.0,0.142857,Women 45-60 | Brand Awareness
4,Instagram,54,0.507546,4.060370,26.0,30.0,0.464286,Men 25-34 | Product Launch
5,Twitter,25,0.456250,3.650000,12.0,15.0,0.444444,Men 25-34 | Product Launch
6,Facebook,47,0.419229,3.353830,21.0,28.0,0.428571,Men 25-34 | Product Launch
7,Pinterest,7,0.053880,0.431036,1.0,8.0,0.111111,Men 25-34 | Product Launch
8,Facebook,38,0.515592,4.124737,20.0,20.0,0.500000,Men 18-24 | Brand Awareness
9,Instagram,46,0.506522,4.052174,23.0,25.0,0.479167,Men 18-24 | Brand Awareness


**De ahí seguimos con la tabla de política de cada arm del bandit**

Esta tabla muestra qué aprendió el agente dentro de cada contexto, brazo por brazo.

Aca ya no solo vemos el canal ganador, sino también:

- cuantas `impresiones` recibió cada canal
- la `recompensa_promedio`
- el `roi_promedio_observado`
- los parámetros `alpha` y `beta`
- la `muestra_esperada`


In [5]:
resultado['comparacion_contextual_vs_global'].head(20)


,Contexto,canal_contextual,recompensa_promedio_contextual,roi_promedio_contextual,canal_global,recompensa_promedio_global,roi_promedio_global,coinciden,mejor_agente
0,Women 45-60 | Brand Awareness,Facebook,0.579684,4.637473,Twitter,0.497152,3.977217,0,contextual
1,Men 25-34 | Product Launch,Instagram,0.507546,4.060370,Twitter,0.497152,3.977217,0,contextual
2,Men 18-24 | Brand Awareness,Facebook,0.515592,4.124737,Twitter,0.497152,3.977217,0,contextual
3,Men 35-44 | Market Expansion,Instagram,0.508710,4.069677,Twitter,0.497152,3.977217,0,contextual
4,Women 25-34 | Increase Sales,Instagram,0.562545,4.500357,Twitter,0.497152,3.977217,0,contextual
5,Women 35-44 | Brand Awareness,Twitter,0.587261,4.698088,Twitter,0.497152,3.977217,1,contextual
6,Men 25-34 | Increase Sales,Instagram,0.545147,4.361176,Twitter,0.497152,3.977217,0,contextual
7,Women 45-60 | Product Launch,Twitter,0.564162,4.513297,Twitter,0.497152,3.977217,1,contextual
8,Women 35-44 | Market Expansion,Instagram,0.568688,4.549500,Twitter,0.497152,3.977217,0,contextual
9,Men 45-60 | Market Expansion,Facebook,0.533581,4.268644,Twitter,0.497152,3.977217,0,contextual


**Finalmente, tenemos la comparación del agente por contexto y el agente global**

Esta es una de las tablas más utiles del notebook, porque compara directamente el agente contextual contra el agente global.

Aca vemos:

- `canal_contextual`
- `canal_global`
- `recompensa_promedio_contextual` y `roi_promedio_contextual`
- `recompensa_promedio_global` y `roi_promedio_global`
- `mejor_agente`

En los resultados que se ven arriba, el agente global termina aprendiendo mucho hacia `Twitter`, mientras que el contextual varía bastante más. Y eso conversa bien con el EDA, porque ya habíamos visto que no hay un único canal ganador para todos los casos.


## Conclusiones

- La evaluación si confirma lo que ya sugeria el EDA: hay heterogeneidad real por contexto y por eso tiene sentido usar un agente contextual.
- El bandit contextual logra una `accuracy_mejor_canal` de `0.75`, lo cual ya es una señal bastante buena para una primera versión del sistema.
- El `roi_promedio_bandit` queda muy cerca del `roi_promedio_optimo`, asi que incluso cuando no acierta exactamente el mejor canal histórico, en promedio se mantiene competitivo.
- El `regret_aproximado_promedio` sale bajo, lo que refuerza la idea de que el modelo está aprendiendo una política razonable.
- La comparación contra el agente global tambien ayuda bastante: el global tiende a simplificar demasiado el problema, mientras que el contextual captura mejor las diferencias entre audiencias y objetivos.
- Tambien se ve que hay algunos contextos donde el global todavía gana. Eso no invalida el enfoque, pero sí sugiere que hay combinaciones donde el contextual aun podría mejorar o donde quizá hay menos datos.

En resumen, con lo que se ve hasta aca, el enfoque contextual con Thompson Sampling sí está justificando su uso.
